# Finetune

In [ ]:
! pip install --upgrade transformers trl accelerate bitsandbytes peft datasets

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import torch

In [3]:
model_id = "Qwen/Qwen3-4B"
data_path = "/kaggle/input/unidata/data.jsonl"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def to_text(example):
    # Convert messages -> single string bằng chat template
    txt = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": txt}

raw = load_dataset("json", data_files=data_path, split="train")
ds = raw.map(to_text, remove_columns=raw.column_names)

In [8]:
print(ds[1]['text'])

<|im_start|>system
Bạn là một AI tư vấn tuyển sinh đại học chuyên nghiệp. Hãy trả lời các câu hỏi một cách chính xác, hữu ích và thân thiện. Có thể sử dụng những thông tin được cung cấp để đưa ra câu trả lời hoặc lời khuyên tốt nhất.<|im_end|>
<|im_start|>user
Câu hỏi: Sinh viên ngành Hóa học Đại học Bách khoa Hà Nội ra trường có tỉ lệ việc làm bao nhiêu?<|im_end|>
<|im_start|>assistant
<think>

</think>

### Tỉ lệ việc làm sinh viên ngành Hóa học (HUST, 2024)  
**Tóm tắt:**  
- Tỉ lệ sinh viên ngành Hóa học có việc làm sau 1 năm tốt nghiệp đạt **90%**.  
- Thời gian tìm việc trung bình khoảng **3 tháng**.  
- Ngành Hóa học nằm trong top ngành có nhu cầu tuyển dụng cao nhất tại HUST.  

| Tình trạng việc làm | Tỉ lệ (%) | Thời gian tìm việc trung bình | 
|---------------------|-----------|-------------------------------| 
| Có việc làm         | 90        | 3 tháng                       | 
| Không có việc làm   | 10        | -                             | 

**Nguồn:** [HUST Tình hình 

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)


In [9]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [10]:
cfg = SFTConfig(
    output_dir="./qwen_lora_sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=2,
    dataset_text_field="text",
    max_length=512,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    bf16=True,
    packing=False,
    report_to="none"
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    peft_config=lora_config,
    args=cfg
)

In [12]:
trainer.train()
trainer.model.save_pretrained("./qwen_lora_adapter")
tokenizer.save_pretrained("./qwen_lora_adapter")

Step,Training Loss
50,0.731900
100,0.392700
150,0.353500
200,0.304900
250,0.294000


('./qwen_lora_adapter/tokenizer_config.json',
 './qwen_lora_adapter/special_tokens_map.json',
 './qwen_lora_adapter/chat_template.jinja',
 './qwen_lora_adapter/vocab.json',
 './qwen_lora_adapter/merges.txt',
 './qwen_lora_adapter/added_tokens.json',
 './qwen_lora_adapter/tokenizer.json')

In [13]:
! zip -r qwen_lora_adapter.zip qwen_lora_adapter

  adding: qwen_lora_adapter/ (stored 0%)
  adding: qwen_lora_adapter/special_tokens_map.json (deflated 63%)
  adding: qwen_lora_adapter/chat_template.jinja (deflated 76%)
  adding: qwen_lora_adapter/adapter_model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 8%)
  adding: qwen_lora_adapter/merges.txt (deflated 57%)
  adding: qwen_lora_adapter/adapter_config.json (deflated 57%)
  adding: qwen_lora_adapter/tokenizer.json (deflated 81%)
  adding: qwen_lora_adapter/vocab.json (deflated 61%)
  adding: qwen_lora_adapter/added_tokens.json (deflated 68%)
  adding: qwen_lora_adapter/README.md (deflated 65%)
  adding: qwen_lora_adapter/tokenizer_config.json (deflated 90%)


# Evaluate

In [ ]:
! pip install --upgrade peft

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

model_id = "Qwen/Qwen3-4B"
adapter_dir = "/kaggle/input/qwen-lora-adapter/qwen_lora_adapter"

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

# 3) attach LoRA adapter
model = PeftModel.from_pretrained(base, adapter_dir)

In [3]:
SYSTEM_INSTRUCTION = "Bạn là một AI tư vấn tuyển sinh đại học chuyên nghiệp. Hãy trả lời các câu hỏi một cách chính xác, hữu ích và thân thiện. Có thể sử dụng những thông tin được cung cấp để đưa ra câu trả lời hoặc lời khuyên tốt nhất."

In [4]:
def generate_content(model, tokenizer, question):
    message = [{"role": "system", "content": SYSTEM_INSTRUCTION},
               {"role": "user", "content": f"Câu hỏi: {question}"}]
    prompt = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs.input_ids.shape[-1]
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0, input_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    return answer

In [5]:
import re

def check_format(answer):
    total_items = 5
    valid_items = 0

    # Tiêu đề
    if bool(re.search(r"^### .+", answer, re.MULTILINE)) == True:
        valid_items += 1

    # Tóm tắt
    if "**Tóm tắt:**" in answer:
        summary_bullets = re.findall(r"^- .+", answer.split("**Bạn có thể hỏi thêm:**")[0], re.MULTILINE)
        if len(summary_bullets) >= 2:
            valid_items += 1

    # Bảng chính
    # Có bảng hay không
    has_table = "|---" in answer
    if has_table:
        total_items += 1
        # kiểm tra có header + ít nhất 1 dòng dữ liệu
        rows = [r for r in answer.splitlines() if "|" in r]
        if len(rows) >= 3:
            valid_items += 1
    else:
        pass

    # Nguồn
    if "**Nguồn:**" in answer:
        valid_items += 1

    # Lưu ý
    if "**Lưu ý:**" in answer:
        valid_items += 1

    # Follow-up
    if "**Bạn có thể hỏi thêm:**" in answer:
        followup_bullets = re.findall(r"^- .+", answer.split("**Bạn có thể hỏi thêm:**")[-1], re.MULTILINE)
        if len(followup_bullets) >= 2:
            valid_items += 1

    return valid_items/total_items

In [6]:
def evaluate(model, tokenizer):
    with open('/kaggle/input/unidata/test_questions.txt', 'r', encoding='utf-8') as file:
        questions = [line.strip() for line in file]
    total_questions = len(questions)
    ratio = 0
    for i, question in enumerate(questions, 1):
        answer = generate_content(model, tokenizer, question)
        ratio += check_format(answer)
        if i % 10 == 0:
            print(f"Processed {i}/{total_questions} - Valid format: {ratio/i:.2%}")
    print(f"Final result: Valid format: {ratio/total_questions:.2%}")

In [7]:
evaluate(model, tokenizer)

Processed 10/100 - Valid format: 96.67%
Processed 20/100 - Valid format: 97.50%
Processed 30/100 - Valid format: 98.33%
Processed 40/100 - Valid format: 98.33%
Processed 50/100 - Valid format: 98.00%
Processed 60/100 - Valid format: 97.50%
Processed 70/100 - Valid format: 97.38%
Processed 80/100 - Valid format: 97.29%
Processed 90/100 - Valid format: 97.22%
Processed 100/100 - Valid format: 97.33%
Final result: Valid format: 97.33%


# Inference

In [ ]:
! pip install --upgrade peft

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

model_id = "Qwen/Qwen3-4B"
adapter_dir = "/kaggle/input/qwen-lora-adapter/qwen_lora_adapter"

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

# 3) attach LoRA adapter
model = PeftModel.from_pretrained(base, adapter_dir)

In [4]:
# 4) build chat prompt from messages using the chat template
messages = [
    {"role": "system", "content": "Bạn là một AI tư vấn tuyển sinh đại học chuyên nghiệp. Hãy trả lời các câu hỏi một cách chính xác, hữu ích và thân thiện. Có thể sử dụng những thông tin được cung cấp để đưa ra câu trả lời hoặc lời khuyên tốt nhất."},
    {"role": "user", "content": f"""Thông tin tham khảo:
Điểm chuẩn vào HUST - Đại học Bách Khoa Hà Nội năm 2025

Năm 2025, Đại học Bách khoa Hà Nội (HUST) tuyển sinh 9.680 chỉ tiêu, với 3 phương thức gồm: phương thức xét tuyển tài năng, phương thức xét tuyển theo điểm thi đánh giá tư duy và phương thức xét tuyển theo điểm thi tốt nghiệp trung học phổ thông. Đại học Bách khoa Hà Nội, đơn vị chủ trì lọc ảo ở miền Bắc, cho biết đã phối hợp cập nhật, hoàn thiện phần mềm, điều chỉnh quy trình và lên phương án kỹ thuật chung cho cả hệ thống.

                                                          Tên ngành    Tổ hợp môn  Điểm chuẩn  Ghi chú
                                  Kỹ thuật Thực phẩm (CT tiên tiến) A00; B00; D07       22.00      NaN
                                   Kỹ thuật sinh học (CT tiên tiến) A00; B00; D07       22.00      NaN
                                                  Kỹ thuật Sinh học A00; B00; D07       24.00      NaN
                                                 Kỹ thuật Thực phẩm A00; B00; D07       24.54      NaN
                                   Kỹ thuật Hóa dược (CT tiên tiến) A00; A01; D07       24.34      NaN
                                                   Kỹ thuật Hóa học A00; B00; D07       24.38      NaN
                                                            Hóa học A00; B00; D07       23.81      NaN
                                                 Công nghệ Giáo dục A00; A01; D01       25.30      NaN
                                                   Quản lý giáo dục A00; A01; D01       24.78      NaN
                 Hệ thống điện và năng lượng tái tạo (CT tiên tiến)      A00; A01       25.80      NaN
                   Kỹ thuật Điều khiển - Tự động hoá (CT tiên tiến)      A00; A01       27.54      NaN
Tin học công nghiệp và Tự động hóa (Chương trình Việt - Pháp PFIEV) A00; A01; D29       26.22      NaN
                                                      Kỹ thuật điện      A00; A01       26.81      NaN
                                  Kỹ thuật điều khiển - Tự động hóa      A00; A01       28.16      NaN
                                Phân tích kinh doanh (CT tiên tiến) A01; D01; D07       25.50      NaN
                 Logistics và Quản lý chuỗi cung ứng (CT tiên tiến) A01; D01; D07       26.06      NaN
                                                 Quản lý năng lượng A00; A01; D01       25.40      NaN
                                                Quản lý Công nghiệp A00; A01; D01       25.60      NaN
                                                Quản trị Kinh doanh A00; A01; D01       25.77      NaN
                                                            Kế toán A00; A01; D01       25.80      NaN
Câu hỏi: Điểm chuẩn HUST 2024"""}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[-1]

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.5,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = outputs[0, input_len:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
print(answer)

### Điểm chuẩn 2024 — HUST (Đại học Bách Khoa Hà Nội)  
**Tóm tắt:**  
- Điểm chuẩn năm 2024 dao động từ **23.81** đến **28.16**.  
- Ngành có điểm chuẩn cao nhất là **Kỹ thuật Điều khiển - Tự động hóa (CT tiên tiến)** với **28.16** điểm.  
- Ngành có điểm chuẩn thấp nhất là **Hóa học** với **23.81** điểm.  

| Ngành                                 | Tổ hợp môn         | Điểm chuẩn | Ghi chú       |
|---------------------------------------|--------------------|------------|---------------|
| Kỹ thuật Điều khiển - Tự động hóa   | A00; A01           | 28.16      |               |
| Kỹ thuật Thực phẩm (CT tiên tiến)   | A00; B00; D07      | 24.54      |               |
| Kỹ thuật Hóa dược (CT tiên tiến)    | A00; A01; D07      | 24.34      |               |
| Kỹ thuật Hóa học                      | A00; B00; D07      | 24.38      |               |
| Quản lý Công nghiệp                   | A00; A01; D01      | 25.60      |               |
| Quản trị Kinh doanh                   | A00; A01;

# Deploy

In [ ]:
! pip install vllm pyngrok
! pip install --upgrade triton

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_KEY = user_secrets.get_secret("NGROK_KEY")

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_KEY)

In [ ]:
import subprocess, os, signal

BASE_MODEL = "Qwen/Qwen3-4B"
LORA_PATH  = "/kaggle/input/qwen-lora-adapter/qwen_lora_adapter"  # đổi path adapter
MAX_LEN    = "2048"

vllm_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", BASE_MODEL,
    "--host", "0.0.0.0", "--port", "8000",
    "--dtype", "float16",
    "--max-model-len", MAX_LEN,
    "--enable-lora",
    "--lora-modules", f"myft={LORA_PATH}",
    "--max-lora-rank", "16",
    # "--enforce-eager",
]

vllm_log = open("/kaggle/working/vllm.log", "w")
vllm_proc = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT, preexec_fn=os.setsid)

print("vLLM started with PID", vllm_proc.pid)


In [ ]:
!tail -n 20 /kaggle/working/vllm.log

In [ ]:
tunnel = ngrok.connect(8000, "http")

print("Public URL:", tunnel.public_url)  # hoặc: str(tunnel)
print("Gọi API: POST", f"{tunnel.public_url}/v1/chat/completions")

In [ ]:
messages = [
    {"role": "system", "content": "Bạn là một AI tư vấn tuyển sinh đại học chuyên nghiệp. Hãy trả lời các câu hỏi một cách chính xác, hữu ích và thân thiện. Có thể sử dụng những thông tin được cung cấp để đưa ra câu trả lời hoặc lời khuyên tốt nhất."},
    {"role": "user", "content": """Thông tin tham khảo:
    Tổng quan
Viện Trí tuệ nhân tạo được thành lập theo quyết định số 162/QĐ-TCCB, ngày 18/03/2022 của Hiệu trưởng Trường Đại học Công nghệ. Tên tiếng Anh: Institute for Artificial Intelligence (IAI). Viện Trí tuệ nhân tạo là đơn vị đào tạo, nghiên cứu, phát triển công nghệ thuộc Trường ĐHCN, hoạt động theo qui chế tổ chức và hoạt động của Viện do Hiệu trưởng Trường ĐHCN ban hành.

Sứ mệnh
Trường ĐHCN xác định sứ mạng của Viện Trí tuệ nhân tạo là “Đào tạo nguồn nhân lực công nghệ chất lượng cao, trình độ cao trong lĩnh vực trí tuệ nhân tạo và các lĩnh vực liên ngành; nghiên cứu phát triển và ứng dụng trí tuệ nhân tạo trong các lĩnh vực đem lại lợi ích xã hội, từ đó đóng góp tích cực vào sự phát triển của Trường ĐHCN, của ĐHQGHN cũng như sự phát triển nền kinh tế và xã hội tri thức của đất nước trong xu thế cuộc cách mạng công nghiệp lần thứ tư”.

Tầm nhìn
Trở thành đơn vị dẫn đầu trong cả nước về đào tạo nguồn nhân lực chất lượng cao ngành trí tuệ nhân tạo, nghiên cứu khoa học và chuyển giao công nghệ liên ngành.

Cơ cấu tổ chức
Văn phòng Viện
Hội đồng khoa học đào tạo
Phòng thí nghiệm Học máy
Phòng thí nghiệm Xử lý ngôn ngữ tự nhiên
Phòng thí nghiệm trọng điểm Hệ thống tích hợp thông minh
Đội ngũ cán bộ
STT	Họ tên	Chức danh	Lĩnh vực	Email
1	TS. Trần Quốc Long	Viện trưởng, Giảng viên chính	Học máy, xử lý hình ảnh	tqlong@vnu.edu.vn
 2	PGS.TS. Nguyễn Việt Hà	Giảng viên cao cấp	Trí tuệ nhân tạo, Học máy, Xử lý ngôn ngữ tự nhiên 	hanv@vnu.edu.vn 
 3	PGS.TS. Nguyễn Phương Thái	Trưởng phòng Thí nghiệm xử lý ngôn ngữ tự nhiên, Giảng viên cao cấp	Xử lý ngôn ngữ tự nhiên	thainp@vnu.edu.vn 
 4	TS. Bùi Ngọc Thăng	Trưởng phòng Thí nghiệm Hệ thống tri thức	Học máy	thangbn@vnu.edu.vn 
5	TS. Nguyễn Thị Ngọc Diệp 	Phó trưởng phòng Thí nghiệm Học máy	Học máy, xử lý hình ảnh	ngocdiep@vnu.edu.vn
6	TS. Trần Hồng Việt	Giảng viên chính	Xử lý ngôn ngữ tự nhiên	 thviet@vnu.edu.vn
7	TS. Triệu Hải Long	Giảng viên	Xử lý ngôn ngữ tự nhiên	thlong@vnu.edu.vn 
8	TS. Bùi Văn Vượng	Giảng viên	Hệ thống tri thức, Toán học	 bui.vuong@vnu.edu.vn
9	TS. Nguyễn Kiêm Hùng	Giảng viên	Các hệ thống tích hợp thông minh, Kiến trúc máy tính	kiemhung@vnu.edu.vn
10	TS. Đỗ Thái Dương	Giảng viên chính	Giải tích phức, Lý thuyết đa thế vị	dtduong@vnu.edu.vn
11	TS. Nguyễn Bích Vân	 Giảng viên chính	 Lý thuyết biểu diễn, đại số kết hợp, ứng dụng đại số trong phương trình vi phân	nbvan@vnu.edu.vn
12	ThS. Quách Công Hoàng	Giảng viên	Các hệ thống tích hợp thông minh, Kiến trúc máy tính	 hoangqc@vnu.edu.vn
13	ThS. Ngô Minh Hương	Giảng viên	Xử lý ngôn ngữ tự nhiên, ngôn ngữ học 	nmhuong@vnu.edu.vn 
14	ThS. Nguyễn Thị Thùy Linh	Giảng viên	 Khai phá quy trình, mô hình ngôn ngữ	linh.nguyen@vnu.edu.vn 
15	ThS. Nguyễn Văn Phi	Giảng viên	Học máy, xử lý hình ảnh	phinv@vnu.edu.vn 
16	CN. Đỗ Thu Uyên	Trợ giảng	Học máy, học tăng cường 	uyendt@vnu.edu.vn 
17	CN. Nguyễn Hải Toàn	Trợ giảng	Học máy	nguyenhaitoan@vnu.edu.vn 
18	CN. Nguyễn Tiến Đạt	Trợ giảng	 Học máy, Xử lý ngôn ngữ tự nhiên	datntien@vnu.edu.vn
19	CN. Lương Sơn Bá	Trợ giảng	Trí tuệ nhân tạo cho Y tế 	bals@vnu.edu.vn
20	CN. Phạm Tiến Du	Trợ giảng	 Học máy	ptdu@vnu.edu.vn
21	CN. Trịnh Ngọc Huỳnh	Trợ giảng	 Trí tuệ nhân tạo cho Y tế, mô hình sinh	huynhtn@vnu.edu.vn
22	CN. Nguyễn Khánh Ly	Chuyên viên Văn phòng Viện	 	lynk@vnu.edu.vn 
23	TS. Hoàng Thanh Tùng	Giảng viên kiêm nhiệm 	Học máy 	 
24	PGS.TS. Lê Anh Cường	Giảng viên kiêm nhiệm 	Học máy 	 
25	TS. Phạm Tiến Lâm	Giảng viên kiêm nhiệm 	Hệ thống tri thức, Khoa học Vật liệu	 
26	TS. Nguyễn Việt Cường	Giảng viên kiêm nhiệm 	Hệ thống máy tính hiệu năng cao	 
27	TS. Trần Tiến Hải	Giảng viên kiêm nhiệm	 	 
28	TS. Trần Văn Khánh	Giảng viên kiêm nhiệm	 	 


Câu hỏi: Viện trí tuệ nhân tạo UET có bao nhiêu giảng viên là Tiến sĩ"""
}
]

In [ ]:
from openai import OpenAI

openai_api_key = "EMPTY"
openai_api_base = "https://f4f9586f58b7.ngrok-free.app/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

chat_response = client.chat.completions.create(
    model="Qwen/Qwen3-4B",
    messages=messages,
    temperature=0.5,
    top_p=0.9,
    extra_body={
        "chat_template_kwargs": {"enable_thinking": False},
    },
)
print(chat_response.choices[0].message.content)

In [ ]:
!ps -ef | grep vllm

In [ ]:
!ps -ef | grep ngrok